# Blend: Run E + n-gram SVM + Nishant's two-stage 8B / 14B

Inference only. Four members, each adds `weight × log p` to one score matrix, then Sinkhorn balancing:

| Member | What | Alone | Blend weight (plain / twin) |
| --- | --- | --- | --- |
| `svc` | 4-view n-gram LinearSVC, fit **here** on `train.csv` + `pool_v1.parquet` | 0.435 CV | 1.021 |
| `lora8b_v1` | Nishant's two-stage Qwen3-8B-Base LoRA (24 layers) | 0.569 CV | 0.226 / 0.079 |
| `lora14b_v1` | Nishant's two-stage Qwen3-14B-Base LoRA (27 layers) | 0.569 CV | 0.429 / 0.150 |
| `run_e` | **Our** two-stage Qwen3-8B-Base LoRA (`job_final` from [`Qwen3-8B-base-Lora-twostage.ipynb`](Qwen3-8B-base-Lora-twostage.ipynb)) | **0.564 LB** | 0.226 / 0.079 |

The first three members with these weights are the public [two-stage notebook](../notebooks/two-stage-lora-on-the-public-pool.ipynb) (LB **0.592**); its weights were fit on out-of-fold predictions. We have no out-of-fold predictions for Nishant's checkpoints, so `run_e` gets the 8B member's weights as a starting point. Run E is the same recipe as `lora8b_v1` with a different seed and setup, so the two will mostly agree; the gain comes from averaging their errors.

Set `enabled=False` on any member in the config cell to submit variants (e.g. turn `run_e` off to reproduce ~0.592, or turn `lora14b_v1` off to save time).

## Attach on Kaggle
1. Competition **Name That LLM**
2. **Qwen3-8B-Base** and **Qwen3-14B-Base** (Kaggle Models, `qwen-lm/qwen-3`)
3. Dataset **`nishantdahal/ntl-two-stage-lora`** (`pool_v1.parquet`, `lora8b_v1/`, `lora14b_v1/`)
4. Your Run E training output with `lora_out/job_final/` (`adapter/`, `readouts.npz`, `norms.safetensors`)
5. **2× T4**, internet off

**Runtime:** ~1 h on the hidden test for the 0.592 members, plus ~15–20 min for `run_e`.

## 0. Config, data

In [ ]:
import gc, glob, json, os, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()
INPUT_DIR = '/kaggle/input'

QWEN3_8B = dict(model_type='qwen3', hidden_size=4096, num_hidden_layers=36)
QWEN3_14B = dict(model_type='qwen3', hidden_size=5120, num_hidden_layers=40)

W_SVC = 1.021
SVC_ENABLED = True
NISHANT = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model
    'lora8b_v1':  dict(enabled=True, w=0.226, twin_w=0.079, bs=16, base=QWEN3_8B),
    'lora14b_v1': dict(enabled=True, w=0.429, twin_w=0.150, bs=8,  base=QWEN3_14B),
}
RUN_E = dict(enabled=True, w=0.226, twin_w=0.079, bs=16, base=QWEN3_8B,
             dir=None,          # set to the job_final path if more than one job_final is attached
             keep=24, layers=[2, 4, 6, 8, 12, 16, 20, 24], max_len=447)
TWIN = dict(thresh=0.85, min_size=2, max_size=18)

# Column order of Nishant's heads (checked against each checkpoint's meta.json)
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f'{INPUT_DIR}/**/{pattern}', recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def logp(P):
    return np.log(np.clip(P, 1e-9, 1))


train = pd.read_csv(find('train.csv'))
test = pd.read_csv(find('test.csv'))
test['text'] = test['text'].fillna('').astype(str)
y = train['model'].map({l: i for i, l in enumerate(LABELS)}).values
assert not np.isnan(y.astype(float)).any()
LOGIT = np.zeros((len(test), K))
MEMBERS = {}   # name -> (n, K) log-probabilities before weighting, saved at the end
print('train', train.shape, 'test', test.shape, 'gpus', torch.cuda.device_count(), flush=True)

## 1. n-gram SVM on train + pool

Four views of each passage: raw characters, words + punctuation, the *shape* of the text (`Xxxx xx x`), and function words only (content words replaced by `CONTENT`). A class-balanced LinearSVC on train + the 34,940 pool passages.

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

FUNCTION_WORDS = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5), ('function', 1, 3, 50000, 0.5)]


def shape_text(t):
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', t)


def function_text(t):
    return ' '.join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else 'CONTENT'
                    for w in re.findall(r'\w+|[^\w\s]', t))


if SVC_ENABLED:
    pool = pd.read_parquet(find('pool_v1.parquet'))
    pool = pool[pool.model.isin(LABELS)]
    fit_texts = train.text.astype(str).tolist() + pool.text.astype(str).tolist()
    fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]
    A, B = [], []
    for view, lo, hi, max_features, weight in VIEWS:
        prep = shape_text if view == 'shape' else function_text if view == 'function' else (lambda s: s)
        analyzer = 'char' if view == 'shape' else 'word' if view == 'function' else view
        opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                    max_features=max_features, dtype=np.float32)
        if analyzer == 'word':
            opts['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
        vec = TfidfVectorizer(**opts)
        A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
        B.append(vec.transform([prep(t) for t in test.text]) * weight)
    svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced').fit(hstack(A, format='csr'), fit_y)
    MEMBERS['svc'] = logp(softmax(svc.decision_function(hstack(B, format='csr'))))
    LOGIT += W_SVC * MEMBERS['svc']
    del A, B, svc, pool, fit_texts
    gc.collect()
print(f'n-gram model done ({time.time() - T0:.0f}s)', flush=True)

## 2. LoRA feature extraction

For each checkpoint: load the first `keep` layers of the base model (split over both T4s if it doesn't fit on one), merge the LoRA into the weights, mean-pool the hidden states at 8 depths (padding and the first token masked out), pass each through its LayerNorm and concatenate.

Nishant's checkpoints were trained with the model's **final RMSNorm switched off**; ours (Run E) kept it. With `keep=24` the last pooled layer is the normed output, so each member uses the setting it was trained with.

In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    for cfg in sorted(glob.glob(f'{INPUT_DIR}/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f'attach the base model {spec}')


def load_trunk(base_dir, keep, dtype, final_norm):
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, 'layer_types', None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    hd = getattr(mc, 'head_dim', None) or mc.hidden_size // mc.num_attention_heads
    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd
               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: '13GiB' for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map='auto', max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to('cuda:0').eval()
    if not final_norm:
        model.norm = nn.Identity()
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg = json.load(open(os.path.join(adapter_dir, 'adapter_config.json')))
    scale = acfg['lora_alpha'] / acfg['r']
    sd = load_file(os.path.join(adapter_dir, 'adapter_model.safetensors'))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if '.lora_A.' in k:
            name = k.split('.lora_A.')[0]
            lin = modules[name[name.index('layers.'):]]
            Bm = sd[k.replace('.lora_A.', '.lora_B.')]
            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))


@torch.no_grad()
def pooled_features(base_dir, adapter_dir, norm_sd, keep, layers, max_len, final_norm, texts, bs, tok_dir=None):
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(tok_dir or base_dir)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, keep, dtype, final_norm)
    merge_lora(model, adapter_dir)
    H = model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to('cuda:0').float()
    norms.load_state_dict(norm_sd)
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=max_len)['input_ids']
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind='stable')
    Z = np.zeros((len(ids), H * len(layers)), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc['attention_mask'].unsqueeze(-1).float().to('cuda:0')
        m[:, 0] = 0
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to('cuda:0').float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()
    del model, norms
    gc.collect()
    torch.cuda.empty_cache()
    assert np.isfinite(Z).all()
    return Z

## 3. Topic clusters for the centred twin

Cluster the test passages by content-word TF-IDF. In clusters of 2–18 passages (likely the same prompt), each LoRA member subtracts the cluster mean from its features and reads them with its centred head. With no clusters this does nothing.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(test.text.tolist())
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(linkage(squareform(D, checks=False), method='average'), TWIN['thresh'], criterion='distance')
    del D
else:
    groups = np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN['min_size']) & (size <= TWIN['max_size'])
print(f'topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18', flush=True)


def add_lora_member(name, Z, plain_w, plain_b, center_w, center_b, w, twin_w, cols=None):
    plain = logp(softmax(Z @ plain_w.T + plain_b))
    twin = None
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(groups[twin_rows]):
            Zc[groups == g] -= Zc[groups == g].mean(0)
        twin = logp(softmax(Zc[twin_rows] @ center_w.T + center_b))
    if cols is not None:   # reorder the checkpoint's classes to LABELS
        plain = plain[:, cols]
        twin = None if twin is None else twin[:, cols]
    MEMBERS[name] = plain
    LOGIT[:] += w * plain
    if twin is not None:
        LOGIT[twin_rows] += twin_w * twin

## 4. Nishant's two-stage checkpoints (8B, 14B)

In [ ]:
for ckpt_name, mc in NISHANT.items():
    if not mc['enabled']:
        continue
    ckpt = os.path.dirname(find(f'{ckpt_name}/meta.json'))
    meta = json.load(open(os.path.join(ckpt, 'meta.json')))
    assert meta['labels'] == LABELS, 'label order differs from the checkpoint heads'
    cfg = meta['cfg']
    heads = load_file(os.path.join(ckpt, 'heads.safetensors'))
    norm_sd = {k[len('norms.'):]: v for k, v in heads.items() if k.startswith('norms.')}
    Z = pooled_features(find_base(mc['base']), os.path.join(ckpt, 'adapter'), norm_sd, cfg['keep'], cfg['layers'],
                        cfg['max_len'], final_norm=False, texts=test.text.tolist(), bs=mc['bs'])
    h = {k: v.float().numpy() for k, v in heads.items() if not k.startswith('norms.')}
    add_lora_member(ckpt_name, Z, h['plain.weight'], h['plain.bias'], h['centered.weight'], h['centered.bias'],
                    mc['w'], mc['twin_w'])
    del Z, heads
    gc.collect()
    print(f'{ckpt_name} done ({time.time() - T0:.0f}s)', flush=True)

## 5. Our Run E checkpoint

Finds the attached `job_final` folder that has `adapter/`, `readouts.npz` and `norms.safetensors`. If several are attached (e.g. Run A and Run E), set `RUN_E['dir']` in the config cell.

In [ ]:
def find_run_e():
    if RUN_E['dir']:
        return Path(RUN_E['dir'])
    hits = []
    for pat in ('/kaggle/input/**/job_final', 'lora_out/job_final'):
        for h in sorted(glob.glob(pat, recursive=True)):
            p = Path(h)
            if (p / 'adapter').is_dir() and (p / 'readouts.npz').exists() and (p / 'norms.safetensors').exists():
                hits.append(p)
    assert hits, 'attach the Run E output (lora_out/job_final)'
    assert len(hits) == 1, f"several job_final folders found, set RUN_E['dir'] to one of: {hits}"
    return hits[0]


if RUN_E['enabled']:
    ckpt = find_run_e()
    with np.load(ckpt / 'readouts.npz', allow_pickle=False) as z:
        classes = [str(x) for x in z['classes'].tolist()]
        assert [int(x) for x in z['pool_layers']] == RUN_E['layers']
        pw, pb = z['plain_coef'].astype(np.float64), z['plain_intercept'].astype(np.float64)
        cw, cb = z['center_coef'].astype(np.float64), z['center_intercept'].astype(np.float64)
    assert sorted(classes) == sorted(LABELS), classes
    cols = [classes.index(l) for l in LABELS]
    nsd = load_file(str(ckpt / 'norms.safetensors'))
    norm_sd = {k.split('norms.', 1)[-1]: v for k, v in nsd.items()}
    tok_dir = str(ckpt / 'tokenizer') if (ckpt / 'tokenizer').is_dir() else None
    Z = pooled_features(find_base(RUN_E['base']), str(ckpt / 'adapter'), norm_sd, RUN_E['keep'], RUN_E['layers'],
                        RUN_E['max_len'], final_norm=True, texts=test.text.tolist(), bs=RUN_E['bs'], tok_dir=tok_dir)
    add_lora_member('run_e', Z, pw, pb, cw, cb, RUN_E['w'], RUN_E['twin_w'], cols=cols)
    del Z
    gc.collect()
    print(f'run_e done from {ckpt} ({time.time() - T0:.0f}s)', flush=True)

## 6. Blend, balance, write `submission.parquet`

Members are combined by summing weighted log-probabilities. Sinkhorn balancing then rescales so each of the 18 models gets ~1/18 of the predictions (skipped on the 3-row local `test.csv`). The agreement table shows how often each member's top pick matches the blend.

In [ ]:
assert MEMBERS, 'no member enabled'
P = softmax(LOGIT)
if len(test) >= 50:
    target = np.full(K, len(P) / K)
    for _ in range(100):
        P *= target / np.clip(P.sum(0), 1e-12, None)
        P /= np.clip(P.sum(1, keepdims=True), 1e-12, None)
pred = P.argmax(1)

sub = pd.DataFrame({'id': test['id'].astype(str), 'model': np.array(LABELS)[pred]})
assert sub['id'].is_unique and set(sub['model']) <= set(LABELS)
sub.to_parquet('submission.parquet', index=False)
np.savez_compressed('blend_members.npz', ids=sub['id'].values, labels=np.array(LABELS),
                    **{k: v.astype(np.float32) for k, v in MEMBERS.items()})

names = list(MEMBERS)
agree = pd.DataFrame([[float((MEMBERS[a].argmax(1) == MEMBERS[b].argmax(1)).mean()) for b in names] for a in names],
                     index=names, columns=names).round(3)
agree['vs_blend'] = [round(float((MEMBERS[n].argmax(1) == pred).mean()), 3) for n in names]
print('top-1 agreement between members:')
print(agree.to_string())
print(sub['model'].value_counts().sort_index().to_string())
print(f'done in {time.time() - T0:.0f}s -> submission.parquet ({len(sub)} rows)')